# Agentic Worksheet: Reach Trajectory Speed Figures

This worksheet is intentionally incomplete. It is designed for practicing agentic coding workflows on a small data-analysis task.

The final goal is to load curated reach data, load a right-hand trajectory, select one or more reaches, and plot the reach path colored by speed.


## How To Use This Worksheet With An Agent

Treat this folder as the whole exercise context. Do not ask the agent to inspect neighboring completed examples, finished notebooks, or solution scripts.

Good agent prompts for this worksheet look like:

- "Read only this notebook and tell me why the next cell fails."
- "Patch the smallest missing piece in the data-loading step."
- "Use only the context in this worksheet and the errors from running cells."
- "Create a short notes file in this folder summarizing the data assumptions before editing code."

The notebook looks close to complete, but several small and medium-sized pieces are missing on purpose.


## Exercise Goal

A completed worksheet should:

- Load one session's reach and trajectory data.
- Check that the data are usable before plotting.
- Let the user choose one reach or all reaches.
- Plot the reach path with speed shown by color.
- Save the figure outputs in this worksheet folder.

The notebook is intentionally missing some of the details needed to do that. Let the errors and nearby variable names guide the debugging process.


## 1. Imports And Constants

This cell is complete. Run it first.


In [ ]:
from pathlib import Path
import re

from matplotlib.collections import LineCollection
from matplotlib import colors as mcolors
import numpy as np

RESULT_NAMES = {
    2: "grabbed",
    3: "missed",
    4: "dropped",
    5: "stalled",
}

HAND_POS_NAMES = {
    0: "unspecified",
    1: "left",
    2: "right",
    3: "above",
    4: "below",
}

POINTS_PER_FRAME_PAIR = 8


## 2. User Settings

This cell is mostly complete. `SESSION_FOLDER` is intentionally blank because this worksheet should not point at a finished example folder.


In [ ]:
worksheet_folder = Path.cwd()
if worksheet_folder.name != "example-worksheet" and (worksheet_folder / "example-worksheet").is_dir():
    worksheet_folder = worksheet_folder / "example-worksheet"

# TODO: Fill in this setup step or ask your agent to improve it.
SESSION_FOLDER = None

# Use "auto" unless the session contains both fixed-cam and legacy-cam trajectory outputs.
WORKSPACE_FORMAT = "auto"  # Options: "auto", "fixed", "legacy"

# Try "single" first. Later, ask the agent to support "all" well.
PLOT_MODE = "single"  # Options: "single", "all"
REACH_SELECTION = 1    # Used when PLOT_MODE is "single".

# The 3D path contains an intentional import bug later in the worksheet.
PLOT_VIEW = "2d"      # Options: "2d", "3d"

# This style setting can be revisited when the plotting cells are debugged.
COLOR_MAP = "speed_heatmap"

output_folder = worksheet_folder / "worksheet-figures"

print(f"Worksheet folder: {worksheet_folder}")
print(f"Session folder setting: {SESSION_FOLDER}")


## 3. Load Data

This is the first intentionally missing logical step.

A useful agent task is to complete this cell using only this worksheet and the errors produced by the following cells. The agent should not copy from a completed solution file.

Broad guidance:

- First make the notebook produce the variables expected by later cells.
- Keep any fix small enough that a beginner can review it.
- If real data are unavailable, make that limitation explicit before adding demo data.


In [ ]:
# TODO: Replace these placeholders with actual loading logic.
curated_reach_rows = None
trajectory_xyz_speed = None
speed_units = None
session_label = None

if curated_reach_rows is None or trajectory_xyz_speed is None:
    raise NotImplementedError(
        "Data loading is intentionally missing. Ask the agent to complete this cell "
        "from the notebook context, without reading finished solution files."
    )


## 4. Validate Curated Reach Rows

This cell is almost complete. One validation check is intentionally missing.


In [ ]:
required_reach_fields = ["frame", "max_delta", "duration", "result", "hand_pos"]

if not isinstance(curated_reach_rows, list):
    raise TypeError("curated_reach_rows should be a list of dictionaries.")

if len(curated_reach_rows) == 0:
    raise ValueError("No curated reaches were loaded.")

clean_reaches = []

for row_number, row in enumerate(curated_reach_rows, start=1):
    for field in required_reach_fields:
        if field not in row:
            raise ValueError(f"Reach row {row_number} is missing required field: {field}")

    frame = int(row["frame"])
    max_delta = int(row["max_delta"])
    duration = int(row["duration"])
    result = int(row["result"])
    hand_pos = int(row["hand_pos"])

    if frame < 0:
        raise ValueError(f"Reach row {row_number} has a negative frame number.")
    if max_delta < 0:
        raise ValueError(f"Reach row {row_number} has a negative max_delta.")
    if duration <= 0:
        raise ValueError(f"Reach row {row_number} has a non-positive duration.")
    if max_delta > duration:
        raise ValueError(f"Reach row {row_number} has max_delta greater than duration.")

    # TODO: Add one missing validation check here.

    if hand_pos not in HAND_POS_NAMES:
        raise ValueError(f"Reach row {row_number} has an unknown hand_pos code: {hand_pos}")

    clean_reaches.append({
        "frame": frame,
        "max_delta": max_delta,
        "duration": duration,
        "result": result,
        "hand_pos": hand_pos,
        "source_row": row_number,
    })

print(f"Validated {len(clean_reaches)} curated reach rows.")


## 5. Sort Reaches And Check For Overlap

This cell is complete. It creates a simple reach index that users can select later.


In [ ]:
clean_reaches = sorted(clean_reaches, key=lambda reach: reach["frame"])

previous_reach = None
for reach in clean_reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    if previous_reach is not None:
        previous_end_frame = previous_reach["frame"] + previous_reach["duration"]
        if reach["frame"] <= previous_end_frame:
            raise ValueError(
                "Curated reaches overlap: "
                f"row {previous_reach['source_row']} and row {reach['source_row']}."
            )
    previous_reach = reach

for reach_index, reach in enumerate(clean_reaches, start=1):
    reach["index"] = reach_index

print("Curated reaches:")
for reach in clean_reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    print(
        f"  {reach['index']:>3}. frame={reach['frame']} "
        f"end={reach_end_frame} "
        f"result={RESULT_NAMES.get(reach['result'], 'unknown')} "
        f"hand_pos={HAND_POS_NAMES[reach['hand_pos']]}"
    )


## 6. Validate The Trajectory Array

This cell is complete if the data-loading step created `trajectory_xyz_speed` correctly.


In [ ]:
trajectory_xyz_speed = np.asarray(trajectory_xyz_speed)

if trajectory_xyz_speed.ndim != 2:
    raise ValueError("trajectory_xyz_speed should be a 2D NumPy array.")

if trajectory_xyz_speed.shape[1] != 4:
    raise ValueError(
        "trajectory_xyz_speed should have exactly four columns: X, Y, Z, speed. "
        f"Found shape {trajectory_xyz_speed.shape}."
    )

if trajectory_xyz_speed.shape[0] < 2:
    raise ValueError("trajectory_xyz_speed has fewer than two frames.")

if not np.isfinite(trajectory_xyz_speed).all():
    raise ValueError("trajectory_xyz_speed contains non-finite values.")

print(f"Trajectory shape: {trajectory_xyz_speed.shape}")
print(f"Speed units: {speed_units}")


## 7. Summarize Loaded Data

This checkpoint should give the user confidence that the notebook loaded the expected session. One small reporting piece is missing.


In [ ]:
loaded_data_summary = [
    f"Session: {session_label}",
    f"Reach rows: {len(clean_reaches)}",
    f"Trajectory frames: {trajectory_xyz_speed.shape[0]}",
]

# TODO: Add one more useful checkpoint line.
extra_summary_line = None

if extra_summary_line is None:
    raise NotImplementedError("Add one short summary line before continuing.")

loaded_data_summary.append(extra_summary_line)

for line in loaded_data_summary:
    print(line)


## 8. Add Reach Details

This step prepares a small table that is useful for review. It contains a subtle mismatch for learners to diagnose from the data already in memory.


In [ ]:
reach_detail_rows = []

for reach in clean_reaches:
    peak_frame = reach["start_frame"] + reach["max_delta"]
    end_frame = reach["frame"] + reach["duration"]
    reach_detail_rows.append({
        "index": reach["index"],
        "frame": reach["frame"],
        "peak_frame": peak_frame,
        "end_frame": end_frame,
        "result": RESULT_NAMES.get(reach["result"], "unknown"),
    })

for row in reach_detail_rows[:5]:
    print(row)


## 9. Select Reaches To Plot

This cell has a missing single-reach branch. It is a good small edit for an agent.


In [ ]:
plot_mode = str(PLOT_MODE).strip().lower()

if plot_mode == "all":
    selected_reaches = clean_reaches
elif plot_mode == "single":
    selected_value = int(REACH_SELECTION)

    # TODO: Finish the single-reach selection branch.
    selected_reaches = []
else:
    raise ValueError("PLOT_MODE should be 'single' or 'all'.")

if len(selected_reaches) == 0:
    raise NotImplementedError(
        "Single-reach selection is intentionally incomplete. "
        "Ask the agent to finish the smallest missing branch."
    )

print(f"Selected reach indices: {[reach['index'] for reach in selected_reaches]}")


## 10. Extract Reach Segments

This cell is complete. It slices the trajectory array for each selected reach.


In [ ]:
reach_segments = []

for reach in selected_reaches:
    start_frame = reach["frame"]
    end_frame = reach["frame"] + reach["duration"]

    if end_frame >= trajectory_xyz_speed.shape[0]:
        raise ValueError(
            f"Reach {reach['index']} ends at frame {end_frame}, "
            f"but trajectory has only {trajectory_xyz_speed.shape[0]} frames."
        )

    segment = trajectory_xyz_speed[start_frame:end_frame + 1, :]
    reach_segments.append({"reach": reach, "segment": segment})

print(f"Prepared {len(reach_segments)} reach segment(s).")


## 11. Calculate A Simple Reach Metric

This step adds a small numeric summary for each selected reach. The metric calculation is intentionally left unfinished.


In [ ]:
reach_metrics = []

for item in reach_segments:
    reach = item["reach"]
    segment = item["segment"]

    # TODO: Calculate a useful path-length value from the segment.
    path_length_mm = None
    max_speed = float(np.max(segment[:, 3]))

    if path_length_mm is None:
        raise NotImplementedError("Add a simple path-length calculation before plotting.")

    reach_metrics.append({
        "index": reach["index"],
        "path_length_mm": path_length_mm,
        "max_speed": max_speed,
    })

reach_metrics


## 12. Plot Reach Paths

The 2D path is almost complete. The 3D path has an intentional package-name error so users can practice debugging an import failure.


In [ ]:
plot_view = str(PLOT_VIEW).strip().lower()

all_segment_speeds = np.concatenate([item["segment"][:, 3] for item in reach_segments])
speed_min = float(np.min(all_segment_speeds))
speed_max = float(np.max(all_segment_speeds))
if speed_min == speed_max:
    speed_max = speed_min + 1.0

if not session_label:
    session_label = "worksheet_session"

if plot_mode == "all":
    title_reach_text = f"all curated reaches (n={len(reach_segments)})"
else:
    selected = reach_segments[0]["reach"]
    title_reach_text = f"reach {selected['index']} frame {selected['frame']}"

figure_title = f"{session_label} | {plot_view.upper()} | {title_reach_text}"

if plot_view == "2d":
    import matplotlib.pyplot as plt

    speed_norm = mcolors.Normalize(vmin=speed_min, vmax=speed_max)
    fig, axis = plt.subplots(figsize=(7, 6))
    color_source = None

    for reach_number, item in enumerate(reach_segments):
        segment = item["segment"]
        y_values = segment[:, 1]
        z_values = segment[:, 2]
        speed_values = segment[:, 3]

        points = np.column_stack([y_values, z_values])
        smooth_points = []
        smooth_speeds = []

        for index in range(points.shape[0] - 1):
            for step in range(POINTS_PER_FRAME_PAIR):
                fraction = step / POINTS_PER_FRAME_PAIR
                smooth_points.append(points[index] + fraction * (points[index + 1] - points[index]))
                smooth_speeds.append(speed_values[index] + fraction * (speed_values[index + 1] - speed_values[index]))

        smooth_points.append(points[-1])
        smooth_speeds.append(speed_values[-1])
        smooth_points = np.asarray(smooth_points)
        smooth_speeds = np.asarray(smooth_speeds)

        segments = np.stack([smooth_points[:-1], smooth_points[1:]], axis=1)
        segment_speeds = (smooth_speeds[:-1] + smooth_speeds[1:]) / 2.0

        collection = LineCollection(segments, cmap=COLOR_MAP, norm=speed_norm, linewidth=2.0)
        collection.set_array(segment_speeds)
        axis.add_collection(collection)
        color_source = collection

        start_label = "start" if reach_number == 0 else None
        end_label = "end" if reach_number == 0 else None
        axis.scatter(y_values[0], z_values[0], color="black", s=24, marker="o", label=start_label, zorder=3)
        axis.scatter(y_values[-1], z_values[-1], color="black", s=24, marker="s", label=end_label, zorder=3)

    axis.autoscale()
    axis.set_aspect("equal", adjustable="datalim")
    axis.set_xlabel("Y position (mm)")
    axis.set_ylabel("Z position (mm)")
    axis.set_title(figure_title)
    axis.legend(loc="best")

    colorbar = fig.colorbar(color_source, ax=axis, shrink=0.8)
    colorbar.set_label(f"Speed ({speed_units})")
    fig.tight_layout()

elif plot_view == "3d":
    # TODO: Something in this 3D setup is intentionally wrong.
    import plotly.graph_object as go

    fig = go.Figure()
    raise NotImplementedError("The 3D plotting body is intentionally unfinished after the import is fixed.")

else:
    raise ValueError("PLOT_VIEW should be '2d' or '3d'.")

fig


## 13. Save The Figure

This cell has one missing filename-detail step. It should save a PNG and PDF for 2D plots once the earlier cells work.


In [ ]:
output_folder.mkdir(exist_ok=True)

safe_session_label = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(session_label)).strip("_")
if not safe_session_label:
    safe_session_label = "worksheet_session"

# TODO: Make this filename more informative.
output_stem = f"{safe_session_label}_{plot_view}"

if plot_view == "2d":
    png_file = output_folder / f"{output_stem}.png"
    pdf_file = output_folder / f"{output_stem}.pdf"
    fig.savefig(png_file, dpi=300)
    fig.savefig(pdf_file)
    print("Saved files:")
    print(f"  {png_file}")
    print(f"  {pdf_file}")
else:
    raise NotImplementedError("Saving for 3D HTML plots is intentionally unfinished.")


## 14. Write Run Notes

This final step should leave a short human-readable note about what was run and what was produced. The note text is intentionally missing.


In [ ]:
notes_file = output_folder / "worksheet_run_notes.md"

# TODO: Create a short note that would help another learner understand this run.
run_notes = None

if run_notes is None:
    raise NotImplementedError("Write a short run note before saving this file.")

notes_file.write_text(run_notes, encoding="utf-8")
print(f"Saved notes: {notes_file}")


## Reflection Prompts

After you finish the worksheet, ask:

- Which errors were easiest for the agent to diagnose?
- Which missing steps needed more context from you?
- Did the agent stay inside this worksheet folder?
- Did the agent make small patches, or did it try to rewrite too much?
- What context file would have helped the agent before editing code?
